**Swiggy Data Analysis**

**1) Business Objective**

To assess and improve the quality, consistency, and completeness of the grocery product listing data by identifying anomalies, inconsistencies, missing information, duplicates, and pricing irregularities across products, categories, stores, and locations, thereby ensuring accurate product information and enhancing the customer shopping experience.

**2) Import Libraries**

In [1]:
import pandas as pd        # For data manipulation
import numpy as np      # For numerical computations
import seaborn as sns  # For advanced visualization
import matplotlib.pyplot as plt # For plotting
import re # For regular expressions

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


**3) Import Dataset**

In [11]:
df = pd.read_csv("/home/user/Downloads/2026-07-06/swiggy_instamart_2026_07_04.csv")  #Reading Dataset from csv file

**4) Data Insights**

In [3]:
print("INFORMATION ABOUT DATASET")
print("--------------------------------")
df.info() 

INFORMATION ABOUT DATASET
--------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1242276 entries, 0 to 1242275
Data columns (total 31 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   crawl_date_and_time      1242276 non-null  object 
 1   platform_name            1242276 non-null  object 
 2   store_id                 1242276 non-null  int64  
 3   store_name_location      1242276 non-null  object 
 4   city                     1242276 non-null  object 
 5   pin_code                 1242276 non-null  int64  
 6   unique_id                1242276 non-null  object 
 7   product_id_upc_ean       1242276 non-null  object 
 8   department               1242276 non-null  object 
 9   category                 1242276 non-null  object 
 10  sub_category             1242276 non-null  object 
 11  product_type             0 non-null        float64
 12  brand_name               1242106 non-nu

In [5]:
print("\nDESCRIPTIVE STATISTICS OF DATASET")
print("--------------------------------")
df.describe().T.round(2) 


DESCRIPTIVE STATISTICS OF DATASET
--------------------------------


,count,mean,std,min,25%,50%,75%,max
store_id,1242276.0,1376207.23,92436.58,762551.0,1386817.0,1403034.0,1403178.0,1405018.0
pin_code,1242276.0,464832.40,161959.25,110007.0,400078.0,500010.0,560094.0,832110.0
product_type,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_description,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
package_type,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
net_quantity,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_mrp,1242276.0,378.08,562.74,1.0,115.0,235.0,450.0,49900.0
item_selling_price,1242276.0,300.57,439.26,1.0,95.0,190.0,364.0,43900.0
product_price,1242276.0,300.57,439.26,1.0,95.0,190.0,364.0,43900.0
unit_price,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print("\nDATASET SHAPE")
print("--------------------------------")
print("The shape =", df.shape)
num_rows, num_cols = df.shape
num_features = num_cols - 1
num_data = num_rows * num_cols

# Print the information about the dataset
print(f"Number of Rows: {num_rows}")
print(f"Number of Columns: {num_cols}")
print(f"Number of Features: {num_features}")
print(f"Number of All Data: {num_data}")


DATASET SHAPE
--------------------------------
The shape = (1242276, 31)
Number of Rows: 1242276
Number of Columns: 31
Number of Features: 30
Number of All Data: 38510556


In [7]:
print("COLUMNWISE DATA PROFILING")
print("--------------------------------")
summary = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    null_percent = round((null_count / len(df)) * 100, 2)
    distinct_count = df[col].nunique(dropna=True)
    
    # Check if all non-null values are unique
    is_unique = "Yes" if df[col].is_unique and null_count == 0 else "No"


    summary.append({
        'column_name': col,
        'null_count': null_count,
        #'null_percent': null_percent,
        'distinct_count': distinct_count,
        'is_unique': is_unique
    })

summary_df = pd.DataFrame(summary)

# Display the summary
print(summary_df)

COLUMNWISE DATA PROFILING
--------------------------------
                column_name  null_count  distinct_count is_unique
0       crawl_date_and_time           0            5694        No
1             platform_name           0               1        No
2                  store_id           0             125        No
3       store_name_location           0             100        No
4                      city           0              47        No
5                  pin_code           0              92        No
6                 unique_id           0         1007557        No
7        product_id_upc_ean           0           31731        No
8                department           0               9        No
9                  category           0              16        No
10             sub_category           0             148        No
11             product_type     1242276               0        No
12               brand_name         170            2929        No
13             pr

In [11]:
df.store_id.unique()
print(len(df.store_id.unique()))
print(len(df.store_name_location.unique()))

125
100


In [14]:
store_mapping = (
    df.groupby('store_name_location')['store_id']
      .nunique()
      .reset_index(name='unique_store_ids')
      .sort_values('unique_store_ids', ascending=False)
)

multi_store = store_mapping[store_mapping['unique_store_ids'] > 1]

print(multi_store)
print("\nNumber of store_name_location with multiple store_ids:", len(multi_store))

                                  store_name_location  unique_store_ids
99  Zepto KK Nagar, PV Rajamannar Salai, Opp. To R...                 2
24  CS Guest House, Mogappair East, 1/8, Bazaar Rd...                 2
78  Reliance Smart SuperStore, No 6/7, Ground Floo...                 2
77  Reddy Nilaya, 44, Appareddy Palya Extension Ro...                 2
58  MRPL R R Petroleum, JP Nagar 9th Phase, J. P. ...                 2
30  D-Mart, Dmart Yelahaka Chikka Bommasandra, Nor...                 2
76  Real Deepak Punjabi Dhaba, NAD Junction, Sanje...                 2
71  Piles Clinic In Indore | Piles Treatment | Fis...                 2
70  Pawan Nivas, Near RamDev Medical, Lake View Re...                 2
38  Fraghill (House Of Luxury Fragrances), Plot 60...                 2
68  Panchsheel Green 1, 39, Panchsheel Greens, Bha...                 2
41  GR Queens Pride, Koppa Rd, Suraksha Nagar, Yel...                 2
44  Greenwoods Extension Tower, Plot No. 1, Inform...           

***Insights :***
_______________
    * Shape of the dataset is 12,42,276 X 31
    * None of the columns is having unique values.
    * 'product_type', 'product_description', 'package_type', 'net_quantity', 'unit_price' are completely null columns.
    * 'brand_name' is a partialy null column with 170 null values.
    * Distinct store_id and store_name_location count are 125 and 100 respectively. While ivestigating, we found that 28 store_name_locations have exactly two store_ids each.
    


**5) Primary key Analysis**

As per the above informations, no column can be choose as primary key. 

In [6]:
# Check if the combination is unique
is_unique = not df.duplicated(subset=['unique_id', 'store_id','product_id_upc_ean', 'product_name']).any()

print("Is (unique_id, store_id, product_id_upc_ean, product_name) combination unique?", is_unique)

Is (unique_id, store_id, product_id_upc_ean, product_name) combination unique? False


Checked several combinations, but couldn't fix a unique combination

**6) Duplicate Analysis**

In [7]:
print("EXACT DUPLICATE ROWS")
print("--------------------------------")
exact_duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)


print("\nDUPLICATE PRODUCTS IN THE SAME STORE")
print("--------------------------------")
product_duplicate_count = df.duplicated(
    subset=['store_id', 'product_name']
).sum()

print(f"\nNumber of duplicate products (store_id + product_name): {product_duplicate_count}")


print("\n========== DUPLICATE ANALYSIS SUMMARY ==========")
print(f"Total Rows                  : {len(df)}")
print(f"Exact Duplicate Rows        : {exact_duplicate_count}")
print(f"Product Duplicate Rows      : {product_duplicate_count}")
print("================================================")

EXACT DUPLICATE ROWS
--------------------------------
Exact duplicate rows: 0

DUPLICATE PRODUCTS IN THE SAME STORE
--------------------------------

Number of duplicate products (store_id + product_name): 295222

========== DUPLICATE ANALYSIS SUMMARY ==========
Total Rows                  : 1242276
Exact Duplicate Rows        : 0
Product Duplicate Rows      : 295222


In [12]:
# checking for duplicate combinations of store_name_location, product_name, product_grammage, item_selling_price and package_type
dup_products = (
    df.groupby([
        'store_name_location',
        'product_name',
        'product_grammage',
        'item_selling_price',
        'package_type'
    ])
    .size()
    .reset_index(name='count')
)

dup_products = dup_products[dup_products['count'] > 1]\
                    .sort_values('count', ascending=False)

print("Duplicate combinations:", len(dup_products))
print(dup_products.head(20))

Duplicate combinations: 0
Empty DataFrame
Columns: [store_name_location, product_name, product_grammage, item_selling_price, package_type, count]
Index: []


In [36]:
# Check for products mapped to multiple categories
product_category_check = (
    df.groupby(['store_name_location', 'product_name','product_grammage','package_type'])['category']
      .nunique()
      .reset_index(name='category_count')
)

# Products mapped to more than one category
inconsistent_products = product_category_check[
    product_category_check['category_count'] > 1
].sort_values('category_count', ascending=False)

print("Number of inconsistent products:",
      len(inconsistent_products))

print(inconsistent_products.head())
print("\nNumber of inconsistent products:", len(inconsistent_products))

Number of inconsistent products: 0
Empty DataFrame
Columns: [store_name_location, product_name, product_grammage, package_type, category_count]
Index: []

Number of inconsistent products: 0


In [37]:
# Check for products mapped to multiple categories
product_category_check = (
    df.groupby(['store_name_location', 'product_name','product_grammage','package_type'])['sub_category']
      .nunique()
      .reset_index(name='sub_category_count')
)

# Products mapped to more than one category
inconsistent_products = product_category_check[
    product_category_check['sub_category_count'] > 1
].sort_values('sub_category_count', ascending=False)

print("Number of inconsistent products:",
      len(inconsistent_products))

print(inconsistent_products.head())
print("\nNumber of inconsistent products:", len(inconsistent_products))

Number of inconsistent products: 0
Empty DataFrame
Columns: [store_name_location, product_name, product_grammage, package_type, sub_category_count]
Index: []

Number of inconsistent products: 0


In [32]:
# Count distinct product names for each unique_id
product_name_counts = (
    df.groupby("unique_id")["product_name"]
      .nunique(dropna=False)
)

# unique_id values having more than one product_name
duplicate_unique_ids = product_name_counts[product_name_counts > 1].index

print(f"Number of unique_id values with different product_name: {len(duplicate_unique_ids)}")
print(duplicate_unique_ids.tolist())

Number of unique_id values with different product_name: 0
[]


***Insights :***
___________________

    * No exact duplicate columns exists on the dataset, but products duplicates on same stores
    * While deeply investigating, these duplicate products differ by grammage, price or package_type.
    * Also confirmed no products duplicates on same store under different categories.
    * In overall, the dataset is not containing duplicate products visibly.


7) MISSING VALUE ANALYSIS

As mentioned some columns are completely null, that not need to be evaluated. brand_name column is partially null with 170 null values. Lets evaluate that.

In [19]:
# Missing summary by store
print("\nMISSING SUMMARY BY STORE")
print("--------------------------------")
missing_by_store = df.groupby('store_name_location')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_store)



MISSING SUMMARY BY STORE
--------------------------------
                                  store_name_location  brand_name
0   1, Dattani Park Rd, Thakur, Dattani Park, Thak...    0.011654
1   101, Lal Bahadur Shastri Rd, Ram Rahim Udyog N...    0.027772
2   11, Pratap Vihar, Ghaziabad, Uttar Pradesh, 20...    0.017409
3   15, Lalbagh Fort Rd, Kalasipalya, Bengaluru, K...    0.029689
4   2, Santara Magan Place 2 Rd, Hulimavu, Bengalu...    0.014245
..                                                ...         ...
95  Urbando Kosmos, 95/4B1A, M.R Radha Rd, Pudupak...    0.007853
96  Viraj Reality Atrium Appt, Pathardi Rd, Chetan...    0.008324
97  Vishwakiran Apartments, Subhash Nagar, Bandra ...    0.025497
98  Zentai Workforce Pvt Ltd, 2nd Floor, RTO Offic...    0.007454
99  Zepto KK Nagar, PV Rajamannar Salai, Opp. To R...    0.015599

[100 rows x 2 columns]


In [21]:
# Missing summary by category
print("\nMISSING SUMMARY BY CATEGORY")
print("--------------------------------")
missing_by_category = df.groupby('category')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_category)


MISSING SUMMARY BY CATEGORY
--------------------------------
                          category  brand_name
0               Atta, Rice and Dal    0.128379
1                        Baby Care    0.000000
2                    Bath and Body    0.000000
3               Biscuits and Cakes    0.000000
4            Cereals and Breakfast    0.000000
5               Chips and Namkeens    0.000000
6                       Chocolates    0.000000
7           Cold Drinks and Juices    0.000000
8            Dairy, Bread and Eggs    0.001414
9                 Feminine Hygiene    0.000000
10                       Hair Care    0.000000
11  Ice Creams and Frozen Desserts    0.041633
12                          Makeup    0.047942
13                     Paan Corner    0.000000
14                        Skincare    0.000000
15                    Sweet Corner    0.000000


In [22]:
print("\nMISSING SUMMARY BY CITY")
print("--------------------------------")

missing_by_city = df.groupby('city')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_city)


MISSING SUMMARY BY CITY
--------------------------------
             city  brand_name
0       Ahmedabad    0.019848
1           Anand    0.010677
2        Attingal    0.000000
3       Bangalore    0.014921
4        Benaulim    0.006927
5          Bhopal    0.000000
6     Bhubaneswar    0.008283
7         Bhugaon    0.022931
8         Chennai    0.017043
9    Choornikkara    0.016865
10        Cuttack    0.000000
11       Dehradun    0.008758
12          Delhi    0.018616
13      Ghaziabad    0.017409
14      Gorakhpur    0.021333
15        Gurgaon    0.015212
16      Hyderabad    0.025754
17         Indore    0.010097
18         Jaipur    0.018165
19     Jamshedpur    0.018013
20        Kolkata    0.017203
21       Kottayam    0.000000
22       Lonavala    0.025219
23        Lucknow    0.000000
24    Madhurawada    0.000000
25        Madurai    0.010627
26         Meerut    0.000000
27         Mohali    0.009574
28         Mumbai    0.013083
29   Muvattupuzha    0.010508
30         N

***Insights :***
_______________

    * In store-wise, missing values are almost equally distributed.
    * In category-wise also mising percentage is below 0.2, which is negliglible.
    * In city-wise too, the missing percentage has no hike at any particular city.
    * So in overall missing% doesn't have any direct impact form store/category/city.

**8) Category Hierarchy Consistency**

In [25]:
print("\n DEPARTMENT - CATEGORY CONSISTENCY")
print("--------------------------------")

dept_category_map = df.groupby('department')['category'].unique()

for dept, cats in dept_category_map.items():
    print("\nDEPARTMENT:", dept)
    print(list(cats))



 DEPARTMENT - CATEGORY CONSISTENCY
--------------------------------

DEPARTMENT: Atta, Rice & Dal
['Atta, Rice and Dal']

DEPARTMENT: Baby Care
['Baby Care']

DEPARTMENT: Bakery & Biscuits
['Biscuits and Cakes']

DEPARTMENT: Beauty & Cosmetics
['Hair Care', 'Makeup', 'Bath and Body']

DEPARTMENT: Cold Drinks & Juices
['Cold Drinks and Juices']

DEPARTMENT: Dairy & Breakfast
['Cereals and Breakfast', 'Dairy, Bread and Eggs']

DEPARTMENT: Munchies
['Chips and Namkeens', 'Cereals and Breakfast']

DEPARTMENT: Personal Care
['Bath and Body', 'Skincare', 'Feminine Hygiene', 'Makeup', 'Hair Care']

DEPARTMENT: Sweet Tooth
['Chocolates', 'Ice Creams and Frozen Desserts', 'Sweet Corner', 'Paan Corner']


In [4]:
print("\n CATEGORY - SUBCATEGORY CONSISTENCY")
print("--------------------------------")

cat_subcat_map = df.groupby('category')['sub_category'].unique()

for cat, subcats in cat_subcat_map.items():
    print("\nCATEGORY:", cat)
    print(list(subcats))


 CATEGORY - SUBCATEGORY CONSISTENCY
--------------------------------

CATEGORY: Atta, Rice and Dal
['Other Flours', 'Millets & Daliya', 'Toor, Moong and Urad', 'Poha & Puffed Rice', 'Rajma, Chola and Others', 'Rice', 'Besan, Sooji and Maida', 'Atta']

CATEGORY: Baby Care
['Baby Wipes', 'Gifts & More', 'Baby Bathing', 'Baby Cream & Lotions', 'Baby Pharma', 'Books and Toys', 'Baby Diapers', 'Baby Hygiene', 'Mom Care', 'Feeding and Teething Needs', 'Baby Oral Care', 'Baby Oil and Talc', 'Clothes & Accessories', 'Baby Food and Formula', 'Travel Needs & Baby Gears']

CATEGORY: Bath and Body
['Soaps', 'Roll on', 'Body Lotion & Oils', 'Face Care', 'Fragrance & Talc', 'Lip care', 'Womens Perfume', 'Bath & Beauty gifts', 'Handwash', 'Mens Perfume', 'Multi groomers', 'Shower gel', 'Bath Accessories', 'Oral care']

CATEGORY: Biscuits and Cakes
['Cakes & Pies', 'Salted & Plain', 'Baking Ingredients', 'Gift Boxes', 'Cookies', 'Cream Biscuits', 'Marie & Digestive', 'Gourmet collection', 'Healthy Sn

***Insights :***
___________________

    * No specific inconsistencies noticed accross department-category or category-subcategory combinations

**9) Suspicious Value Checking**

In [12]:
def find_suspicious_values(df, column_name):
    suspicious_values = [
        'null', 'none', 'n/a', 'na', 'nil', 'not available',
        'not applicable', '-', '--', '0', ''
    ]
    
    mask = (
        df[column_name].isna() |
        df[column_name].astype(str).str.strip().str.lower().isin(suspicious_values) |
        df[column_name].str.fullmatch(r'\d+') |   # numeric only
        (df[column_name].str.len() < 3)    # too short
        )
    
    df_suspicious = df[mask]
    print(f"Number of suspicious values in '{column_name}': {len(df_suspicious)}")
    return df_suspicious[column_name].unique()

print(find_suspicious_values(df, 'product_name'))
print(find_suspicious_values(df, 'brand_name'))
print(find_suspicious_values(df, 'store_name_location'))
print(find_suspicious_values(df, 'department'))
print(find_suspicious_values(df, 'category'))
print(find_suspicious_values(df, 'sub_category'))

Number of suspicious values in 'product_name': 0
[]
Number of suspicious values in 'brand_name': 3438
[nan 'RG' '921' 'Yu' 'He' 'Go' 'Jo' 'P' 'SS' 'C4' 'Ob' 'Lu' 'SP']
Number of suspicious values in 'store_name_location': 0
[]
Number of suspicious values in 'department': 0
[]
Number of suspicious values in 'category': 0
[]
Number of suspicious values in 'sub_category': 0
[]


***Insights :***
____________

    * Suspicious values noticed on brand_name column.
    * Checked website to confirm whether these brand_names really exists or not.
    * Most of them really exists, but urls of some brands are not able to access.
    * Complete urls on brand_name 'RG' and 'C4' and some urls on other brands are not accessible, which is a critical issue.

**10) Price Anomalies**

In [13]:
def check_negative_and_zero(df, column):
    """
    Returns rows where the given column has:
    - negative values (< 0)
    - zero values (== 0)
    """

    # Ensure numeric (safe for real-world messy data)
    df[column] = pd.to_numeric(df[column], errors='coerce')

    negative_values = df[df[column] < 0]
    zero_values = df[df[column] == 0]

    return {
        "negative_values": negative_values[['unique_id', column]],
        "zero_values": zero_values[['unique_id', column]]
    }

print("\nItem MRP:")
print(check_negative_and_zero(df, 'item_mrp'))
print("\nItem Selling Price:")
print(check_negative_and_zero(df, 'item_selling_price'))
print("\nProduct Price:")
print(check_negative_and_zero(df, 'product_price'))
print("\nPromo Offer Price:")
print(check_negative_and_zero(df, 'promo_offer_price'))


Item MRP:
{'negative_values': Empty DataFrame
Columns: [unique_id, item_mrp]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, item_mrp]
Index: []}

Item Selling Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, item_selling_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, item_selling_price]
Index: []}

Product Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, product_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, product_price]
Index: []}

Promo Offer Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, promo_offer_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, promo_offer_price]
Index: []}


In [14]:
# check if item_selling_price is greater than item_mrp
df[df['item_selling_price'] > df['item_mrp']][['unique_id', 'item_selling_price', 'item_mrp']]

,unique_id,item_selling_price,item_mrp


In [15]:
# check if discount_in_percent is greater than 0 and item_selling_price is greater than or equal to item_mrp
df[
    (df['discount_in_percent'] > 0) &
    (df['item_selling_price'] >= df['item_mrp'])
][['unique_id', 'item_selling_price', 'item_mrp', 'discount_in_percent']]

,unique_id,item_selling_price,item_mrp,discount_in_percent


In [16]:
# check if promo_offer_description is not null and promo_offer_price is null
df[
    (df['promo_offer_description'].notnull()) &

    (df['promo_offer_price'].isnull())][['unique_id', 'promo_offer_description', 'promo_offer_price']]

,unique_id,promo_offer_description,promo_offer_price


In [17]:
# check if promo_offer_price is not null and promo_offer_price is not equal to item_selling_price
df[
    (df['promo_offer_price'].notnull()) &
    (df['promo_offer_price'] != df['item_selling_price'])
][['unique_id', 'promo_offer_description', 'promo_offer_price', 'item_selling_price']]

,unique_id,promo_offer_description,promo_offer_price,item_selling_price


In [18]:
# check if out_of_stock_flag is True and stock_count is greater than 0
df[
    (df['out_of_stock_flag'] == True) &
    (df['stock_count'] > 0)
][['unique_id', 'store_id', 'product_id_upc_ean']]

,unique_id,store_id,product_id_upc_ean


***Insights :***
____________

    * No specific anomalies found on price related columns
    * All such columns are greater than 0.
    * selling price and mrp are equal.
    * selling price is always less than mrp if discount percentage exists.
    * if promo offer description exists, promo offer price also exists.
    * If out of stock flag true, stock count is zero.

**11) URL Checking**


In [19]:
total_records = len(df)
missing_urls = df['product_url'].isnull().sum()
unique_urls = df['product_url'].nunique(dropna=True)
duplicate_urls = df['product_url'].duplicated().sum()

print("URL ANALYSIS SUMMARY")
print("-" * 40)
print(f"Total records           : {total_records}")
print(f"Missing URLs            : {missing_urls}")
print(f"Unique URLs             : {unique_urls}")
print(f"Duplicate URLs          : {duplicate_urls}")
print(f"Uniqueness %            : {(unique_urls/total_records)*100:.2f}%")

URL ANALYSIS SUMMARY
----------------------------------------
Total records           : 1242276
Missing URLs            : 0
Unique URLs             : 31731
Duplicate URLs          : 1210545
Uniqueness %            : 2.55%


In [20]:
# Duplicate URL Records
# ----------------------------
duplicate_url_df = df[
    df['product_url'].duplicated(keep=False)
].sort_values('product_url')

print("\nDuplicate URL Records:")
print(duplicate_url_df[['unique_id', 'product_name', 'product_url']].head())


Duplicate URL Records:
         unique_id                                       product_name  \
708428  MF0GD8AP1D  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
195233  9OFFIH2VYA  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
923913  5JNTTWBHZ8  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
537730  AI3H650VGN  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
732945  KF2U4FJISY  FACES CANADA Ultime Pro Splash Nail Enamel - P...   

                                             product_url  
708428  https://www.swiggy.com/instamart/item/0009OYBIMZ  
195233  https://www.swiggy.com/instamart/item/0009OYBIMZ  
923913  https://www.swiggy.com/instamart/item/0009OYBIMZ  
537730  https://www.swiggy.com/instamart/item/0009OYBIMZ  
732945  https://www.swiggy.com/instamart/item/0009OYBIMZ  


In [21]:
# ----------------------------
# URL Format Validation
# ----------------------------

# Regex for HTTP/HTTPS URLs
url_pattern = re.compile(
    r'^(https?://)'               # http:// or https://
    r'([\w.-]+)'                  # domain
    r'(\.[a-zA-Z]{2,})'           # TLD
    r'([/\w .?%&=+-]*)?$'         # path/query
)

invalid_urls = df[
    df['product_url'].notnull() &
    ~df['product_url'].astype(str).str.match(url_pattern)
]

print("\nInvalid URL Format Count:", len(invalid_urls))

print("\nSample Invalid URLs:")
print(
    invalid_urls[
        ['unique_id', 'product_name', 'product_url']
    ].head(10)
)


Invalid URL Format Count: 0

Sample Invalid URLs:
Empty DataFrame
Columns: [unique_id, product_name, product_url]
Index: []


In [22]:
# ----------------------------
# URLs without http/https
# ----------------------------
missing_protocol = df[
    df['product_url'].notnull() &
    ~df['product_url'].astype(str).str.startswith(('http://', 'https://'))
]

print("\nURLs missing protocol (http/https):", len(missing_protocol))

print(
    missing_protocol[
        ['unique_id', 'product_name', 'product_url']
    ].head()
)


URLs missing protocol (http/https): 0
Empty DataFrame
Columns: [unique_id, product_name, product_url]
Index: []


***Insights***
___________
* Format and pattern of urls are correct
* But 1210545 duplicate urls present on the dataset.
* However these duplcates have different store location and different price range.

**12) City, Store, Pincode Analysis**

In [23]:
# Missing percentage by city

missing_by_city = (
    df.groupby('city')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by city

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_city = (
    df.groupby('city')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by city

discount_by_city = (
    df.groupby('city')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by city

products_by_city = (
    df.groupby('city')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
city_analysis = (
    missing_by_city
    .merge(stockout_by_city, on='city', how='outer')
    .merge(discount_by_city, on='city', how='outer')
    .merge(products_by_city, on='city', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nCITY ANALYSIS SUMMARY")
print("--------------------------------")
print(city_analysis)

/tmp/ipykernel_3402/439306221.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



CITY ANALYSIS SUMMARY
--------------------------------
             city  missing_perc  stockout_perc  avg_discount_perc  \
16      Hyderabad     16.129863      40.178025          16.294209   
22       Lonavala     16.129846      41.039005          16.609617   
35      Pune City     16.129787      35.788981          17.308179   
7         Bhugaon     16.129772      30.879768          16.957961   
14      Gorakhpur     16.129720      53.504000          17.520853   
0       Ahmedabad     16.129673      53.862948          17.202352   
31    Navi Mumbai     16.129649      34.192402          16.727494   
43       Thrissur     16.129648      40.936650          17.158877   
12          Delhi     16.129633      32.916341          16.466920   
18         Jaipur     16.129618      33.115350          17.730245   
19     Jamshedpur     16.129613      30.811492          17.638746   
13      Ghaziabad     16.129594      30.217038          16.490657   
20        Kolkata     16.129587      42.835025 

In [27]:
# Missing percentage by store

missing_by_store = (
    df.groupby('store_name_location')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by store

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_store = (
    df.groupby('store_name_location')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by store

discount_by_store = (
    df.groupby('store_name_location')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by store

products_by_store = (
    df.groupby('store_name_location')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
store_analysis = (
    missing_by_store
    .merge(stockout_by_store, on='store_name_location', how='outer')
    .merge(discount_by_store, on='store_name_location', how='outer')
    .merge(products_by_store, on='store_name_location', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nSTORE ANALYSIS SUMMARY")
print("--------------------------------")
print(store_analysis)

/tmp/ipykernel_3402/3222998141.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



STORE ANALYSIS SUMMARY
--------------------------------
                                  store_name_location  missing_perc  \
42  GVR Villa, 19, Durga Enclave Road No 12 Banjar...     15.626376   
24  CS Guest House, Mogappair East, 1/8, Bazaar Rd...     15.626030   
19  Babes N Tots PreSchool Vejalpur, Gate Of Shrin...     15.626018   
64  North NCL Lane, N NCL LN, Ruby Block, Satyam E...     15.626007   
27  Choice Dry Fruits And Chocolates, Shop No P 3,...     15.625969   
..                                                ...           ...   
20  Best Bakery, Palayam Airport Rd, Pettah, Thiru...     15.625000   
57  Like Me Family Beauty Hub, Companymukku NIT, K...     15.625000   
47  Halvala Nikhil's Home, 1-1-123/6/b, Prashanthn...     15.625000   
56  Lakshmi Pearl, 108, Dharmapuri Colony, Pothina...     15.625000   
60  Mahakal Dhaba, Nayta Mundla, Indore, Madhya Pr...     15.625000   

    stockout_perc  avg_discount_perc  product_count  
42      53.099196          15.720687

In [25]:
# Missing percentage by pincode

missing_by_pincode = (
    df.groupby('pin_code')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by pincode

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_pincode = (
    df.groupby('pin_code')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by pincode

discount_by_pincode = (
    df.groupby('pin_code')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by pincode

products_by_pincode = (
    df.groupby('pin_code')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
pincode_analysis = (
    missing_by_pincode
    .merge(stockout_by_pincode, on='pin_code', how='outer')
    .merge(discount_by_pincode, on='pin_code', how='outer')
    .merge(products_by_pincode, on='pin_code', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nPINCODE ANALYSIS SUMMARY")
print("--------------------------------")
print(pincode_analysis)

/tmp/ipykernel_3402/3770454122.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



PINCODE ANALYSIS SUMMARY
--------------------------------
    pin_code  missing_perc  stockout_perc  avg_discount_perc  product_count
47    500034     15.626376      53.099196          15.720687           7309
72    600040     15.626030      39.333993          16.198813           9708
16    380051     15.626018      55.817490          17.005975           7359
48    500100     15.626007      36.973504          16.417009          10059
32    410210     15.625969      37.673073          16.356479           7748
..       ...           ...            ...                ...            ...
8     201303     15.625000      26.431828          17.604807           9657
12    250004     15.625000      21.530705          17.547731           9772
41    452020     15.625000      41.378958          17.662819           7824
43    462044     15.625000      29.806763          17.131079           4878
59    560048     15.625000      37.310914          16.184114          13759

[92 rows x 5 columns]


***Insights :***
__________________

    * missing percentage is almost equally distributed over all dimensions
    * stockout percentage and product count has significant hike based on some cities, stores, pincodes etc.
    * However discount_percentage is also have almost equal distribution over different dimensions

**13) High Price Variations**

In [28]:
# Price Variation Analysis
price_variation = (
    df.groupby('product_id_upc_ean')
      .agg(
          product_name=('product_name', 'first'),
          min_price=('item_selling_price', 'min'),
          max_price=('item_selling_price', 'max'),
          mean_price=('item_selling_price', 'mean'),
          num_locations=('city', 'nunique')
      )
      .reset_index()
)

# Calculate percentage variation
price_variation['price_variation_pct'] = (
    (price_variation['max_price'] - price_variation['min_price'])
    / price_variation['min_price'] * 100
).round(2)

# Flag products having >30% variation
high_variation = price_variation[
    price_variation['price_variation_pct'] > 30
].sort_values('price_variation_pct', ascending=False)

print(high_variation.head(20))

      product_id_upc_ean                                       product_name  \
12655         DYU1NS9OBF                                 Nutella B ready T1   
29200         X1UP6TZ2CH   CLEAR Premium Drinking Water with Added Minerals   
27628         V5U7GYM8HO       RENEE Pink False Stick On Nails (Pack of 24)   
15309         GY12SME90J  NOICE Coconut Water With Malai Chunks (No Pres...   
22014         OFELFK1MN3  NOICE Blueberry Burst Fudge Whey Protein Bar (...   
5621          63FRSPU0EC     NOICE Natural Coconut Water (No Preservatives)   
13388         ESWAQTN109             Himalayan Natural Mineral Water Bottle   
24854         RUR8E3DZ69                              Red Bull Energy Drink   
20021         M9QXZYJMLV  Red Bull Energy Drink Watermelon The Red Editi...   
14620         G5SH3SUHS1     Red Bull Energy Drink, The Green Edition 250ml   
11701         CWA1JGHGHP     Red Bull Energy Drink - Yellow Edition, 250 ml   
17243         J3VWEOK5FN  Philips Avent Electric Bre

In [29]:
# check if any product has more than one selling price with same grammage across different stores
price_variation = (
    df.groupby(['product_name', 'product_grammage'])['item_selling_price']
      .nunique()
      .reset_index(name='price_count')
)

# Keep only products having more than one selling price
price_variation = price_variation[
    price_variation['price_count'] > 1
]

print(price_variation)

                                            product_name product_grammage  \
2      100% Whole Wheat Bread 400GM & Pintola All Nat...          1 Combo   
3                                      10X Classic Besan             1 kg   
4                                      10X Classic Besan            500 g   
5                          10X Classic Chakki Fresh Atta            10 kg   
6                          10X Classic Chakki Fresh Atta             5 kg   
...                                                  ...              ...   
38189  plum green tea alcohol free toner and cleansin...          1 combo   
38190  plum green tea oil free moisturiser and free t...          1 combo   
38192                      skin & strands Glycerine Soap            125 g   
38201                               uncle john Lemon Pie           450 ml   
38204                               uncle john Pistachio           450 ml   

       price_count  
2                4  
3                6  
4           

In [30]:
# Group by product_name and product_grammage and find price varation statistics
price_analysis = (
    df.groupby(['product_name', 'product_grammage'])['item_selling_price']
      .agg(['count', 'min', 'max', 'mean', 'std'])
      .reset_index()
)

# Calculate price difference and percentage variation
price_analysis['price_diff'] = (
    price_analysis['max'] - price_analysis['min']
)

price_analysis['pct_variation'] = (
    (price_analysis['price_diff'] / price_analysis['mean']) * 100
).round(2)

# Flag products with >20% variation
high_variation = price_analysis[
    price_analysis['pct_variation'] > 20
].sort_values('pct_variation', ascending=False)

print(high_variation)

                                            product_name product_grammage  \
7630   Conscious Chemist 2% Salicylic Acid Face Moist...         50 g x 2   
13879  HealthFields Organic Rozana Toor Split (Arhar ...             1 kg   
4081                         Beyond Water Peach Iced Tea       250 ml x 2   
5699    CLEAR Premium Drinking Water with Added Minerals       200 ml x 4   
33529  THE NATURIK CO High Protein Millet Pancake & W...            150 g   
...                                                  ...              ...   
23586                        Nandini GoodLife Toned Milk           500 ml   
7694   Cookie Man Choco Chunk Cookies & Sugar free Mu...          1 Combo   
25533  PINQ POLKA Hot Pink Sanitary Pad Box (XL) (Pac...        20 pieces   
27449  Polka Pop Orange Sparkling Water & Polka Pop L...          1 Combo   
31968  Soulflower Rosemary Hair Serum with Redensyl, ...            30 ml   

       count  min   max        mean         std  price_diff  pct_variation 

In [31]:
# check if any product has more than one selling price with same grammage on same store
price_variation = (
    df.groupby(['store_name_location', 'product_name', 'product_grammage'])['item_selling_price']
      .agg(['min', 'max', 'nunique', 'count'])
      .reset_index()
)

# Keep only products having more than one unique selling price
price_variation = price_variation[
    price_variation['nunique'] > 1
]

# Calculate price difference
price_variation['price_difference'] = (
    price_variation['max'] - price_variation['min']
)

print(price_variation.sort_values('price_difference', ascending=False))

                                      store_name_location  \
873099  Shiv General Store, Block D, Ardee City, Secto...   
82241   528/1, Shitala Mata Mandir Marg, Sheetla Colon...   
674958  Panchsheel Green 1, 39, Panchsheel Greens, Bha...   
70147   3, 2nd Cross Rd, AECS Layout 1st Stage, Sanjay...   
901779  Shree Sai EnterPrises, Dombala, Panvel, Navi M...   
...                                                   ...   
279572  Cult Fit, 2nd Floor, 17/N, 18th Cross Rd, Sect...   
70388   3, 2nd Cross Rd, AECS Layout 1st Stage, Sanjay...   
279574  Cult Fit, 2nd Floor, 17/N, 18th Cross Rd, Sect...   
846688  Samar Fabrications, 20/A, Oddarapalya, Kengeri...   
765593  Real Deepak Punjabi Dhaba, NAD Junction, Sanje...   

                                             product_name product_grammage  \
873099                         Lindt Napolitains Assorted            350 g   
82241                          Lindt Napolitains Assorted            350 g   
674958                         Li

***Insights :***
_________________

    * Found price anomalies with high variation of price for the same product with the same grammage unit.
    * Multiple products are there with the same product_name, grammage, store_name, and different offer price.
    * The product_url of such products are too same and not accessible.